# T2.2 — Semantic Mapping Upload to DBRepo

Uploads all semantic mappings from `docs/semantic_mapping.csv` to DBRepo via REST API.

**Owner:** Person B | **Task:** T2.2 — Semantic Mapping | **Dataset:** Hohe Warte Vienna Weather

## Step 0 — Install the official DBRepo Python library

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'dbrepo', '--quiet'])
print('dbrepo library ready')


dbrepo library ready


## Step 1 — Configuration

In [2]:
import os
from getpass import getpass
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

DOTENV_PATH = find_dotenv()
load_dotenv(DOTENV_PATH, override=True)
REPO_ROOT = Path(DOTENV_PATH).resolve().parent if DOTENV_PATH else Path.cwd()

ENDPOINT = os.getenv("DBREPO_ENDPOINT", "https://test.dbrepo.tuwien.ac.at")
DATABASE_ID = os.getenv("DBREPO_DATABASE_ID")
USERNAME = os.getenv("DBREPO_USERNAME") or input("DBRepo username: ")
PASSWORD = os.getenv("DBREPO_PASSWORD") or getpass("DBRepo password: ")

if not USERNAME:
    raise RuntimeError("DBREPO_USERNAME is not configured.")
if not PASSWORD:
    raise RuntimeError("DBREPO_PASSWORD is not configured.")
if not DATABASE_ID:
    raise RuntimeError("DBREPO_DATABASE_ID is not configured.")

MAPPING_FILE = REPO_ROOT / "docs" / "semantic_mapping.csv"
print(f"Mapping file found: {MAPPING_FILE.resolve()}")

Mapping file found: /Users/kerimhalilovic/Documents/GitHub/Vienna-Weather-Wet-Month-Prediction/docs/semantic_mapping.csv


## Step 2 — Connect to DBRepo and fetch database structure

The API requires **table UUID** and **column UUID** — not names. We fetch them here.

In [3]:
from dbrepo.RestClient import RestClient

client = RestClient(endpoint=ENDPOINT, username=USERNAME, password=PASSWORD)

db = client.get_database(database_id=DATABASE_ID)
print(f"Connected to database: {db.name}")

tables = db.tables or []
print(f"Tables found: {[t.name for t in tables]}")
if not tables:
    raise RuntimeError("No tables found — check DATABASE_ID or ensure T2.1 is complete")


Connected to database: vienna_weather_wet_months
Tables found: ['weather_measurement_v2', 'weather_measurement', 'time_dimension', 'station']


## Step 3 — Build name-to-ID lookup maps

In [37]:
table_id_map  = {} 
column_id_map = {}

for table in tables:
    table_id_map[table.name] = table.id
    if not table.columns:
        table = client.get_table(database_id=DATABASE_ID, table_id=table.id)
    print(f"  Table '{table.name}' -> {table.id} ({len(table.columns)} columns)")
    for col in table.columns:
        column_id_map[(table.name, col.name)] = col.id

print(f"\nMapped {len(table_id_map)} tables, {len(column_id_map)} columns")
if len(column_id_map) == 0:
    raise RuntimeError("No columns found — cannot upload. Check DBRepo schema.")

print("\nDiscovered columns in DBRepo:")
for k in sorted(column_id_map.keys()):
    print(f"  {k[0]}.{k[1]}")


  Table 'weather_measurement_v2' -> 3674fea3-a7be-4dfe-8356-bc692bd1ff6c (26 columns)
  Table 'weather_measurement' -> 631c878e-1f39-47a0-be38-0b3c0e2733c8 (26 columns)
  Table 'time_dimension' -> fa248a2c-bfb6-4d8e-a89b-2dbd19ab8cde (3 columns)
  Table 'station' -> ab02386c-e27c-4c1f-a27d-93034ce3fa79 (8 columns)

Mapped 4 tables, 63 columns

Discovered columns in DBRepo:
  station.altitude_m
  station.district_code
  station.latitude_deg
  station.longitude_deg
  station.nuts_code
  station.station_name
  station.station_num
  station.sub_district_code
  time_dimension.ref_month
  time_dimension.ref_year
  time_dimension.time_id
  weather_measurement.mean_t_max_c
  weather_measurement.mean_t_min_c
  weather_measurement.measurement_id
  weather_measurement.num_clear
  weather_measurement.num_cloud
  weather_measurement.num_frost
  weather_measurement.num_heat
  weather_measurement.num_ice
  weather_measurement.num_precp_01
  weather_measurement.num_summer
  weather_measurement.num_win

## Step 4 — Load the semantic mapping CSV

In [38]:
import csv

mappings = []
with open(MAPPING_FILE, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        mappings.append(row)

print(f"Loaded {len(mappings)} mappings from CSV")
for m in mappings[:3]:
    print(m)


Loaded 37 mappings from CSV
{'table_name': 'weather_measurement_v2', 'column_name': 'measurement_id', 'ontology_uri': 'http://purl.org/dc/terms/identifier', 'ontology_label': 'Measurement identifier'}
{'table_name': 'weather_measurement_v2', 'column_name': 'station_num', 'ontology_uri': 'http://purl.org/dc/terms/identifier', 'ontology_label': 'Station identifier'}
{'table_name': 'weather_measurement_v2', 'column_name': 'time_id', 'ontology_uri': 'http://purl.org/dc/terms/identifier', 'ontology_label': 'Time identifier'}


## Step 5 — Upload each semantic concept using the official dbrepo client

In [41]:
from json.decoder import JSONDecodeError
import requests

def get_table_json(database_id, table_id):
    url = f"{ENDPOINT}/api/v1/database/{database_id}/table/{table_id}"
    response = requests.get(
        url,
        auth=(USERNAME, PASSWORD),
        headers={"Accept": "application/json"}
    )
    response.raise_for_status()
    return response.json()


def get_column_json(table_json, column_name):
    for column in table_json.get("columns", []):
        if column.get("name") == column_name:
            return column
    return None


success, warnings, skipped, errors = 0, 0, 0, []

for row in mappings:
    tname = row["table_name"]
    cname = row["column_name"]
    concept_uri = row["ontology_uri"]

    tid = table_id_map.get(tname)
    cid = column_id_map.get((tname, cname))

    if tid is None or cid is None:
        skipped += 1
        print(f"SKIP  {tname}.{cname} — not found in DBRepo schema")
        continue

    try:
        table_json = get_table_json(DATABASE_ID, tid)
        column_json = get_column_json(table_json, cname)

        if column_json is None:
            skipped += 1
            print(f"SKIP  {tname}.{cname} — not found in REST metadata")
            continue

        existing_unit_uri = column_json.get("unit_uri")

        client.update_table_column(
            database_id=DATABASE_ID,
            table_id=tid,
            column_id=cid,
            concept_uri=concept_uri,
            unit_uri=existing_unit_uri
        )

        success += 1
        print(f"OK    {tname}.{cname} -> {concept_uri}")

    except JSONDecodeError:
        warnings += 1
        print(
            f"WARN  {tname}.{cname} -> DBRepo returned empty response body; "
            "will verify via REST metadata later"
        )

    except Exception as e:
        errors.append((tname, cname, str(e)))
        print(f"FAIL  {tname}.{cname} -> {e}")

print()
print(f"Upload attempt result: {success} OK, {warnings} warnings, {skipped} skipped, {len(errors)} failed")

if errors:
    print("\nFailed rows:")
    for e in errors:
        print(f"  {e[0]}.{e[1]}: {e[2]}")

WARN  weather_measurement_v2.measurement_id -> DBRepo returned empty response body; will verify via REST metadata later
WARN  weather_measurement_v2.station_num -> DBRepo returned empty response body; will verify via REST metadata later
WARN  weather_measurement_v2.time_id -> DBRepo returned empty response body; will verify via REST metadata later
WARN  weather_measurement_v2.t_mean_c -> DBRepo returned empty response body; will verify via REST metadata later
WARN  weather_measurement_v2.t_max_c -> DBRepo returned empty response body; will verify via REST metadata later
WARN  weather_measurement_v2.t_min_c -> DBRepo returned empty response body; will verify via REST metadata later
WARN  weather_measurement_v2.mean_t_max_c -> DBRepo returned empty response body; will verify via REST metadata later
WARN  weather_measurement_v2.mean_t_min_c -> DBRepo returned empty response body; will verify via REST metadata later
WARN  weather_measurement_v2.p_mean_hpa -> DBRepo returned empty response 

## Step 6 — Verify: read back spot-checks from DBRepo

In [42]:
verified = 0
missing = 0
mismatched = 0
unit_removed = 0

for row in mappings:
    tname = row["table_name"]
    cname = row["column_name"]
    expected_concept_uri = row["ontology_uri"]

    tid = table_id_map.get(tname)

    if tid is None:
        print(f"FAILED: table {tname} not found")
        missing += 1
        continue

    table_json = get_table_json(DATABASE_ID, tid)
    column_json = get_column_json(table_json, cname)

    if column_json is None:
        print(f"FAILED: {tname}.{cname} not found in REST response")
        missing += 1
        continue

    actual_concept_uri = column_json.get("concept_uri")
    actual_unit_uri = column_json.get("unit_uri")

    if actual_concept_uri == expected_concept_uri:
        print(f"OK: {tname}.{cname}")
        print(f"  concept_uri: {actual_concept_uri}")
        print(f"  unit_uri   : {actual_unit_uri}")
        verified += 1
    else:
        print(f"MISMATCH: {tname}.{cname}")
        print(f"  expected concept_uri: {expected_concept_uri}")
        print(f"  actual concept_uri  : {actual_concept_uri}")
        print(f"  unit_uri            : {actual_unit_uri}")
        mismatched += 1
        
    if cname not in ["nuts_code", "station_name"] and actual_unit_uri is None:
        print(f"WARNING: {tname}.{cname} has no unit_uri after concept upload")
        unit_removed += 1

print("\n===================================")
print("Concept mapping verification summary")
print("===================================")
print(f"Verified concept mappings : {verified}")
print(f"Missing rows/columns      : {missing}")
print(f"Mismatches                : {mismatched}")
print(f"Potential units removed   : {unit_removed}")
print("===================================")

OK: weather_measurement_v2.measurement_id
  concept_uri: http://purl.org/dc/terms/identifier
  unit_uri   : http://www.ontology-of-units-of-measure.org/resource/om-2/one
OK: weather_measurement_v2.station_num
  concept_uri: http://purl.org/dc/terms/identifier
  unit_uri   : http://www.ontology-of-units-of-measure.org/resource/om-2/one
OK: weather_measurement_v2.time_id
  concept_uri: http://purl.org/dc/terms/identifier
  unit_uri   : http://www.ontology-of-units-of-measure.org/resource/om-2/one
OK: weather_measurement_v2.t_mean_c
  concept_uri: http://qudt.org/vocab/quantitykind/Temperature
  unit_uri   : http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement_v2.t_max_c
  concept_uri: http://qudt.org/vocab/quantitykind/Temperature
  unit_uri   : http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement_v2.t_min_c
  concept_uri: http://qudt.org/vocab/quantitykind/Temperature
  unit_uri   : http://www.ontology